In [ ]:
import os
import re
import time
import pandas as pd
import requests
from dotenv import load_dotenv

# === Load environment and GitHub tokens ===
env_path = "All_Tokens.env"
load_dotenv(env_path)
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7) if os.getenv(f"GITHUB_TOKEN_{i}")]
if not tokens:
    raise ValueError("❌ No GitHub tokens found.")
token_index = 0

# === CI Patterns ===
ci_patterns = {
    r'\.travis\.yml$': 'Travis_CI',
    r'\.appveyor\.yml$': 'AppVeyor',
    r'appveyor\.yml$': 'AppVeyor',
    r'circle\.yml$': 'Circle_CI',
    r'\.circleci/config\.yml$': 'Circle_CI',
    r'azure-pipelines\.yml$': 'Azure_Pipelines',
    r'\.github/workflows/.*\.(yml|yaml)$': 'GitHub_Actions',
    r'bitbucket-pipelines\.yml$': 'Bitbucket',
    r'\.gitlab-ci\.yml$': 'GitLab',
    r'Jenkinsfile\.yml$': 'Jenkins',
    r'bitrise\.yml$': 'Bitrise',
    r'bamboo\.yml$': 'Bamboo',
    r'codeship-services\.yml$': 'Codeship',
    r'\.gocd\.yaml$': 'GoCD',
    r'\.cirrus\.yml$': 'Cirrus',
    r'wercker\.yaml$': 'Wercker',
    r'semaphore\.yml$': 'Semaphore',
    r'codemagic\.yaml$': 'Nevercode',
}
# === Paths ===
input_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output.csv"
output_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step4_ci_detection_output.csv"
yml_log_csv = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step4_detected_yml_files.csv"

# === Load input data ===
input_df = pd.read_csv(input_csv)
input_df['html_url'] = input_df['html_url'].astype(str).str.strip()

# Filter: keep all rows but only process Valid_Repo_Step3 == yes
if os.path.exists(output_csv):
    print("🔁 Resuming from previously saved output.")
    input_df = pd.read_csv(output_csv)
else:
    input_df['yml_detected'] = input_df.get('yml_detected', 'none')
    input_df['total_yml_files'] = input_df.get('total_yml_files', 0)

# === Load or initialize log file for matched ymls ===
if os.path.exists(yml_log_csv):
    yml_log = pd.read_csv(yml_log_csv)
    detected_files = yml_log.to_dict("records")
else:
    detected_files = []

# === Filter to valid repos and unprocessed ===
# Work on the full dataset, but isolate which rows to review
input_df['Valid_Repo_Step3'] = input_df['Valid_Repo_Step3'].astype(str).str.lower()
rows_to_review = input_df[(input_df['Valid_Repo_Step3'] == 'yes') & (input_df['yml_detected'] == 'none')].index

print(f"🔍 Starting detection: {len(rows_to_review)} repos to review.")

def get_valid_response(url, headers, verbose=True):
    global token_index
    token_count = len(tokens)
    attempts = 0
    sleep_times = []

    while attempts < token_count:
        token_id = token_index % token_count
        token = tokens[token_id]
        headers['Authorization'] = f'token {token}'
        response = requests.get(url, headers=headers)

        if response.status_code == 200 or response.status_code == 404:
            # Successful or valid not found
            if verbose:
                print(f"✅ Token {token_id + 1} used successfully.")
            return response, token_id + 1

        elif response.status_code == 403 and response.headers.get("X-RateLimit-Remaining") == "0":
            reset_time = int(response.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait_seconds = max(reset_time - int(time.time()), 1)
            if verbose:
                print(f"⏳ Token {token_id + 1} rate-limited. Skipping and will retry others.")
            sleep_times.append(wait_seconds)
            token_index += 1
            attempts += 1
            continue

        else:
            if verbose:
                print(f"⚠️ Unexpected error (HTTP {response.status_code}) on token {token_id + 1}")
            token_index += 1
            attempts += 1
            continue

    # All tokens failed or are rate-limited — sleep for shortest reset
    if sleep_times:
        min_wait = min(sleep_times)
        if verbose:
            print(f"😴 All tokens exhausted. Sleeping for {min_wait} seconds...")
        time.sleep(min_wait)
        return get_valid_response(url, headers, verbose)
    else:
        if verbose:
            print("❌ All tokens failed with non-rate-limit errors.")
        return None, None

# === Process repos ===
for review_counter, idx in enumerate(rows_to_review, start=1):

    url = input_df.at[idx, 'html_url']
    print(f"🔎 [{review_counter}/{len(rows_to_review)}] Checking: {url}")
    try:
        parts = url.rstrip('/').split('/')
        owner, repo = parts[-2], parts[-1]
        headers = {}

        r1, token_used = get_valid_response(f"https://api.github.com/repos/{owner}/{repo}", headers)
        if not r1 or r1.status_code != 200:
            print("❌ Repo info failed.")
            input_df.at[idx, 'yml_detected'] = 'no'
            continue

        default_branch = r1.json().get('default_branch', 'main')

        r2, _ = get_valid_response(f"https://api.github.com/repos/{owner}/{repo}/git/trees/{default_branch}?recursive=1", headers)
        if not r2 or r2.status_code != 200:
            print("❌ File tree failed.")
            input_df.at[idx, 'yml_detected'] = 'no'
            continue

        files = [item['path'] for item in r2.json().get('tree', []) if item['type'] == 'blob']
        matched = []
        for f in files:
            for pattern, ci_type in ci_patterns.items():
                if re.search(pattern, f, re.IGNORECASE):
                    raw_url = f"https://raw.githubusercontent.com/{owner}/{repo}/{default_branch}/{f}"
                    r3, _ = get_valid_response(raw_url, headers)
                    if r3 and r3.status_code == 200:
                        matched.append((f, ci_type))
                        new_log_entry = {
                            "html_url": url,
                            "file_path": f,
                            "ci_type": ci_type,
                        }
                        # Append new log entry immediately
                        pd.DataFrame([new_log_entry]).to_csv(yml_log_csv, mode='a', header=not os.path.exists(yml_log_csv), index=False)

                    break

        input_df.at[idx, 'yml_detected'] = 'yes' if matched else 'no'
        input_df.at[idx, 'total_yml_files'] = len(matched)

        # Save progress after each repo
        input_df.to_csv(output_csv, index=False)
        

    except Exception as e:
        print(f"⚠️ Error processing {url}: {e}")
        continue

print("\n✅ CI YML scan complete. Output and logs saved.")


🔁 Resuming from previously saved output.
🔍 Starting detection: 7359 repos to review.
🔎 [1/7359] Checking: https://github.com/donkingliang/RadarView
✅ Token 1 used successfully.
✅ Token 1 used successfully.
🔎 [2/7359] Checking: https://github.com/nihad92/SwipeableCards
✅ Token 1 used successfully.
✅ Token 1 used successfully.
🔎 [3/7359] Checking: https://github.com/hariprasanths/GoogleNewsStandAnimation-Android
✅ Token 1 used successfully.
✅ Token 1 used successfully.
🔎 [4/7359] Checking: https://github.com/wuyr/FanLayout
✅ Token 1 used successfully.
✅ Token 1 used successfully.
🔎 [5/7359] Checking: https://github.com/PortgasAce/ZoomRecyclerView
✅ Token 1 used successfully.
✅ Token 1 used successfully.
🔎 [6/7359] Checking: https://github.com/lioilwin/AutoInstall
✅ Token 1 used successfully.
✅ Token 1 used successfully.
🔎 [7/7359] Checking: https://github.com/pili-engineering/QNRTC-Android
✅ Token 1 used successfully.
✅ Token 1 used successfully.
🔎 [8/7359] Checking: https://github.com/R